In [1]:
import pennylane as qml
from pennylane import numpy as np

In [2]:
weightage = [1.0,0.8,1.8]
observable = [qml.PauliX(0),qml.PauliX(1),qml.PauliZ(0)@qml.PauliZ(1)]

In [3]:
hamil = qml.Hamiltonian(weightage,observable)
def prepare_state_00():
    pass
def prepare_state_01():
    qml.PauliX(1)
def prepare_state_10():
    qml.PauliX(0)
def prepare_state_11():
    qml.PauliX(0)
    qml.PauliX(1)

In [4]:
states = [prepare_state_00,prepare_state_01,prepare_state_10,prepare_state_11]
def ansatz(theta):
    qml.RY(theta[0],wires=1)
    qml.RX(theta[1],wires=0)
    qml.CNOT(wires=[0,1])
    qml.RY(theta[2],wires=0)
    qml.RX(theta[3],wires=1)
device = qml.device('default.qubit',wires=2)

In [5]:
@qml.qnode(device)
def energy(theta,index):
    states[index]()
    ansatz(theta)
    return qml.expval(hamil)
weights=[4.0,3.0,2.0,1.0]
def cost(theta):
    total=0.0
    for i,w in enumerate(weights):
        total+=w*energy(theta,i)
    return total

In [6]:
np.random.seed(42)
theta = np.random.uniform(0,2*np.pi,4,requires_grad=True)
opt = qml.AdamOptimizer(stepsize=0.2)
steps = 200
for i in range(steps):
    theta,curr = opt.step_and_cost(cost,theta)
    if i%20==0:
        print(f'current theta is {theta} and cost is {curr}')

current theta is [2.55330496 6.17351414 4.39925359 3.56148231] and cost is -2.8839799029935795
current theta is [4.286067   6.32018595 4.39145406 3.16133586] and cost is -5.703142689523441
current theta is [3.67617128 6.26829038 3.98819778 3.1161631 ] and cost is -5.853737897205083
current theta is [3.76660674 6.27662329 4.09153251 3.15077195] and cost is -5.888377210818309
current theta is [3.80927692 6.2837294  4.09609853 3.14393996] and cost is -5.888560861643523
current theta is [3.80204121 6.28286563 4.09479747 3.1422149 ] and cost is -5.888936483244715
current theta is [3.79681173 6.28266993 4.09307307 3.14206216] and cost is -5.889003728852054
current theta is [3.79483723 6.28335639 4.09194676 3.14158644] and cost is -5.889013484119218
current theta is [3.7940387  6.28311731 4.09149253 3.14158027] and cost is -5.8890146395301475
current theta is [3.79387705 6.28318078 4.09150191 3.14157895] and cost is -5.889014656956288


In [8]:
print(f'ground state energy is {energy(theta,0)}')
print(f'1st excited state energy is {energy(theta,1)}')
print(f'2nd excited state energy is {energy(theta,2)}')
print(f'3rd excited state energy is {energy(theta,3)}')
energies=[energy(theta,i) for i in range(4)]
print(energies)

ground state energy is -2.131110312940395
1st excited state energy is 0.504316265438782
2nd excited state energy is -0.5043162652687634
3rd excited state energy is 2.1311103127703763
[tensor(-2.13111031, requires_grad=True), tensor(0.50431627, requires_grad=True), tensor(-0.50431627, requires_grad=True), tensor(2.13111031, requires_grad=True)]


In [9]:
num_qubits = 2
dev = qml.device('default.qubit',wires = num_qubits)

In [10]:
@qml.qnode(dev)
def time_evolution_simulation(t,theta):
    qml.Hadamard(wires = 0)
    qml.Hadamard(wires=1)
    qml.adjoint(ansatz)(theta)
    global_phase = [np.exp(-1j*e*t) for e in energies]
    qml.DiagonalQubitUnitary(global_phase, wires=[0, 1])
    ansatz(theta)
    return qml.state()

In [13]:
time = [1.0,2.8,8.9,9.11]
for t in time:
    states = time_evolution_simulation(t,theta)
    print(np.round(states, 4))
    print()

[-0.4914-0.6975j  0.0676+0.0399j -0.243 -0.2842j  0.316 -0.1618j]

[0.6023+0.1626j 0.2875+0.5376j 0.4624-0.0586j 0.1476+0.0037j]

[ 0.6915-0.0129j  0.2087-0.4885j  0.4769+0.1065j -0.0059+0.0275j]

[0.5767-0.3367j 0.1943-0.5904j 0.4068+0.002j  0.0244-0.0401j]

